# Módulo 2: Análisis Exploratorio de Datos (EDA)
## Clasificación de Conducción Distractiva
### IRNA - Universidad Nacional de Colombia

Este notebook realiza un análisis exploratorio completo del dataset de conducción distractiva,
incluyendo distribución de clases, ejemplos visuales, estadísticas de tamaño y análisis de color.

**Clases del dataset:**
- c0: Conducción segura
- c1: Enviando texto (mano derecha)
- c2: Hablando por teléfono (mano derecha)
- c3: Enviando texto (mano izquierda)
- c4: Hablando por teléfono (mano izquierda)
- c5: Operando radio/controles
- c6: Bebiendo
- c7: Alcanzando hacia atrás
- c8: Arreglándose cabello/maquillaje
- c9: Hablando con pasajero

## 1. Imports y Configuración

In [ ]:
# Importaciones necesarias para el EDA
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, ImageDraw
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para los gráficos
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Directorio de salida para figuras (mismo directorio del notebook)
OUTPUT_DIR = '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Librerías cargadas correctamente')
print(f'NumPy: {np.__version__}')

## 2. Definición de Clases del Dataset

In [ ]:
# Definición de las 5 clases reales de conducción
CLASS_NAMES = {
    0: 'c0: Conducción segura',
    1: 'c1: Girando / Mirando espejos',
    2: 'c2: Texteando al conducir',
    3: 'c3: Hablando por teléfono',
    4: 'c4: Otras actividades'
}

CLASS_SHORT = [f'c{i}' for i in range(5)]

# Colores distintivos por clase para visualización
CLASS_COLORS_VIS = [
    '#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#e67e22'
]

print('Clases del dataset:')
for k, v in CLASS_NAMES.items():
    print(f'  {v}')

## 3. Generador de Imágenes Sintéticas

In [ ]:
def generate_synthetic_driver_images(n_per_class=150, img_size=224):
    """
    Genera imágenes sintéticas de conductores con patrones visuales por clase.
    Cada clase tiene fondo de color distinto + formas geométricas que simulan la distracción.
    
    Args:
        n_per_class (int): número de imágenes por clase
        img_size (int): tamaño de las imágenes en píxeles
    Returns:
        images (list): lista de arrays numpy (H, W, 3)
        labels (list): lista de etiquetas enteras [0-9]
    """
    # Colores base por clase (RGB) - cada clase tiene un tono identificativo
    class_colors = {
        0: (70,  130, 50),   # verde     - conducción segura
        1: (200, 50, 80),   # naranja   - texto mano derecha
        2: (180, 80,  180),  # morado    - teléfono mano derecha
        3: (80,  80,  200),  # azul      - texto mano izquierda
        4: (200, 180, 60),   # amarillo  - teléfono mano izquierda
        5: (50, 200, 200),  # cyan      - radio/controles
        6: (200, 150, 50),  # marrón    - bebiendo
        7: (150, 50, 200),  # violeta   - alcanzando hacia atrás
        8: (255, 150, 200),  # rosa      - cabello/maquillaje
        9: (50, 200, 150),  # verde-azul - conversando con pasajero
    }
    # Número base de formas geométricas por clase (simula actividad visual)
    class_shapes = {0: 5, 1: 15, 2: 12, 3: 15, 4: 12, 5: 8, 6: 5, 7: 7, 8: 18, 9: 6}
    
    images, labels = [], []
    np.random.seed(42)
    
    for cls in range(5):
        base_color = class_colors[cls]
        n_shapes   = class_shapes[cls]
        for i in range(n_per_class):
            # Ligera variación del color de fondo para simular diversidad
            variation = np.random.randint(-20, 20, 3)
            bg = tuple(np.clip(np.array(base_color) + variation, 0, 255).astype(int).tolist())
            img  = Image.new('RGB', (img_size, img_size), bg)
            draw = ImageDraw.Draw(img)
            # Formas geométricas aleatorias para variabilidad intra-clase
            for _ in range(n_shapes + np.random.randint(0, 5)):
                x = np.random.randint(0, img_size)
                y = np.random.randint(0, img_size)
                r = np.random.randint(5, 40)
                c = tuple(np.random.randint(50, 255, 3).tolist())
                st = np.random.randint(0, 3)
                if st == 0:
                    draw.ellipse([x-r, y-r, x+r, y+r], fill=c)
                elif st == 1:
                    draw.rectangle([x-r, y-r, x+r, y+r], fill=c)
                else:
                    draw.line([x-r, y-r, x+r, y+r], fill=c, width=3)
            # Ruido gaussiano para simular variaciones de iluminación y cámara
            arr  = np.array(img).astype(np.float32)
            arr += np.random.normal(0, 15, arr.shape)
            arr  = np.clip(arr, 0, 255).astype(np.uint8)
            images.append(arr)
            labels.append(cls)
    
    return images, labels

print('Función generadora definida correctamente.')

## 4. Carga del Dataset

In [ ]:
# Carga del dataset real de Kaggle
images  = []
labels  = []
data_source = 'Kaggle (Real)'

import kagglehub
print('Cargando dataset de Kaggle real...')
path = r'C:\Users\santy\.cache\kagglehub\datasets\arafatsahinafridi\multi-class-driver-behavior-image-dataset\versions\1\Multi-Class Driver Behavior Image Dataset'
CLASSES = ['safe_driving', 'turning', 'texting_phone', 'talking_phone', 'other_activities']

for cls_idx, cls_name in enumerate(CLASSES):
    folder = os.path.join(path, cls_name)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:400]
        for fname in files:
            img = Image.open(os.path.join(folder, fname)).convert('RGB').resize((224, 224))
            images.append(np.array(img))
            labels.append(cls_idx)

images = np.array(images)
labels = np.array(labels)

print(f'\nFuente de datos : {data_source}')
print(f'Total imágenes  : {len(images)}')
print(f'Shape del array : {images.shape}')
print(f'Tipo de datos   : {images.dtype}')

## 5. Distribución de Clases

In [ ]:
# Conteo de imágenes por clase
class_counts = Counter(labels.tolist())
counts_array = np.array([class_counts[i] for i in range(5)])

print('Distribución de clases:')
print('-' * 50)
total = len(labels)
for cls_id in range(5):
    pct = counts_array[cls_id] / total * 50
    bar = '█' * int(pct / 2)
    print(f'  {CLASS_NAMES[cls_id]:<35} {counts_array[cls_id]:>4} imgs ({pct:5.1f}%) {bar}')
print('-' * 50)
print(f'  Total: {total} imágenes')

In [ ]:
# Gráfico de barras + gráfico circular de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Barras ---
bars = axes[0].bar(CLASS_SHORT, counts_array, color=CLASS_COLORS_VIS,
                   edgecolor='black', linewidth=0.7)
axes[0].set_xlabel('Clase', fontsize=12)
axes[0].set_ylabel('Número de imágenes', fontsize=12)
axes[0].set_title('Distribución de Imágenes por Clase', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts_array):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(count), ha='center', va='bottom', fontsize=10)
mean_val = np.mean(counts_array)
axes[0].axhline(y=mean_val, color='red', linestyle='--', linewidth=2,
                label=f'Media: {mean_val:.0f}')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.5)

# --- Circular ---
wedges, texts, autotexts = axes[1].pie(
    counts_array, labels=CLASS_SHORT, colors=CLASS_COLORS_VIS,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_class_distribution.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_class_distribution.png')

## 6. Ejemplos de Imágenes por Clase (Grid 10×3)

In [ ]:
# Grid 5x3: 3 ejemplos representativos de cada clase
np.random.seed(0)
fig, axes = plt.subplots(5, 3, figsize=(5, 34))
fig.suptitle('Ejemplos de Imágenes por Clase\nDataset de Conducción Distractiva',
             fontsize=15, fontweight='bold', y=1.005)

for cls_id in range(5):
    cls_indices = np.where(labels == cls_id)[0]
    selected    = np.random.choice(cls_indices, min(3, len(cls_indices)), replace=False)
    for col, idx in enumerate(selected):
        ax = axes[cls_id, col]
        ax.imshow(images[idx])
        ax.axis('off')
        if col == 0:
            parts = CLASS_NAMES[cls_id].split(': ')
            ax.set_ylabel(f'{parts[0]}\n{parts[1]}', fontsize=8.5,
                         rotation=0, labelpad=15, va='center', ha='right')
        if cls_id == 0:
            ax.set_title(f'Ejemplo {col+1}', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_examples_grid.png'), bbox_inches='tight', dpi=120)
plt.show()
print('Guardado: fig_examples_grid.png')

## 7. Estadísticas de Tamaño de Imágenes

In [ ]:
# Estadísticas de dimensiones del dataset
heights = images[:, :, :, 0].shape[1] * np.ones(len(images), dtype=int)  # todos iguales post-resize
widths  = images[:, :, :, 0].shape[2] * np.ones(len(images), dtype=int)
# Para datasets reales pueden variar; usamos el array directo
img_heights = [images[i].shape[0] for i in range(len(images))]
img_widths  = [images[i].shape[1] for i in range(len(images))]
img_sizes_kb = [images[i].nbytes / 1024 for i in range(len(images))]

print('='*60)
print('ESTADÍSTICAS DE TAMAÑO DE IMÁGENES')
print('='*60)
print(f'  Altura   - Media: {np.mean(img_heights):.1f} | Min: {np.min(img_heights)} | Max: {np.max(img_heights)}')
print(f'  Anchura  - Media: {np.mean(img_widths):.1f}  | Min: {np.min(img_widths)}  | Max: {np.max(img_widths)}')
print(f'  Canales  - {images[0].shape[2]} (RGB)')
print(f'  Tamaño por imagen: {np.mean(img_sizes_kb):.1f} KB (media)')
print(f'  Tamaño total: {images.nbytes / (1024**2):.1f} MB')
print(f'  Shape del array completo: {images.shape}')
print('='*60)

In [ ]:
# Visualización de estadísticas de tamaño
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Histograma de alturas
axes[0].hist(img_heights, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(np.mean(img_heights), color='red', linestyle='--', linewidth=2,
               label=f'Media: {np.mean(img_heights):.0f} px')
axes[0].set_xlabel('Altura (píxeles)', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].set_title('Distribución de Alturas', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Histograma de anchuras
axes[1].hist(img_widths, bins=20, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].axvline(np.mean(img_widths), color='red', linestyle='--', linewidth=2,
               label=f'Media: {np.mean(img_widths):.0f} px')
axes[1].set_xlabel('Anchura (píxeles)', fontsize=12)
axes[1].set_ylabel('Frecuencia', fontsize=12)
axes[1].set_title('Distribución de Anchuras', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.4)

# Scatter alto vs. ancho coloreado por clase
n_show = min(500, len(images))
scatter_colors = [CLASS_COLORS_VIS[labels[i]] for i in range(n_show)]
axes[2].scatter(img_widths[:n_show], img_heights[:n_show], c=scatter_colors, alpha=0.5, s=15)
axes[2].set_xlabel('Anchura (píxeles)', fontsize=12)
axes[2].set_ylabel('Altura (píxeles)', fontsize=12)
axes[2].set_title('Alto vs. Ancho por Clase', fontsize=13, fontweight='bold')
axes[2].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_size_stats.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_size_stats.png')

## 8. Análisis de Balance de Clases

In [ ]:
# Métricas cuantitativas de balance
mean_count     = np.mean(counts_array)
std_count      = np.std(counts_array)
cv             = std_count / mean_count * 100
imbalance_ratio = counts_array.max() / max(counts_array.min(), 1)

print('='*55)
print('ANÁLISIS DE BALANCE DE CLASES')
print('='*55)
print(f'  Media de imágenes por clase : {mean_count:.1f}')
print(f'  Desviación estándar         : {std_count:.2f}')
print(f'  Coeficiente de variación    : {cv:.2f}%')
print(f'  Ratio de desbalance (max/min): {imbalance_ratio:.2f}x')
print(f'  Clase con MÁS imágenes      : c{counts_array.argmax()} ({counts_array.max()} imgs)')
print(f'  Clase con MENOS imágenes    : c{counts_array.argmin()} ({counts_array.min()} imgs)')
print()
if cv < 10:
    print('  => Dataset BIEN BALANCEADO (CV < 10%)')
elif cv < 25:
    print('  => Dataset MODERADAMENTE BALANCEADO (10% <= CV < 25%)')
else:
    print('  => Dataset DESBALANCEADO (CV >= 25%) — considerar oversampling/pesos por clase')
print('='*55)

In [ ]:
# Visualización de balance: desviaciones + mapa de similitud semántica
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Barras de desviación respecto a la media ---
deviations   = counts_array - mean_count
colors_dev   = ['#e74c3c' if d < 0 else '#2ecc71' for d in deviations]
axes[0].bar(CLASS_SHORT, deviations, color=colors_dev, edgecolor='black', linewidth=0.7)
axes[0].axhline(0, color='black', linewidth=1.5)
axes[0].set_xlabel('Clase', fontsize=12)
axes[0].set_ylabel('Desviación respecto a la media', fontsize=12)
axes[0].set_title('Desbalance por Clase\n(verde = sobre la media, rojo = bajo la media)',
                  fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.4)
for i, dev in enumerate(deviations):
    axes[0].text(i, dev + (0.3 if dev >= 0 else -1.5), f'{dev:+.0f}',
                ha='center', fontsize=9)

# --- Mapa de similitud semántica entre clases ---
# Creado a partir de conocimiento del dominio (similitud esperada de confusión)
similarity = np.full((10, 10), 1.0)
np.fill_diagonal(similarity, 10)
similarity[1, 2] = similarity[2, 1] = 5   # texto vs teléfono mano derecha
similarity[3, 4] = similarity[4, 3] = 5   # texto vs teléfono mano izquierda
similarity[1, 3] = similarity[3, 1] = 4   # texto derecha vs izquierda
similarity[2, 4] = similarity[4, 2] = 4   # teléfono derecha vs izquierda
similarity[7, 8] = similarity[8, 7] = 3   # alcanzar vs arreglarse
similarity[5, 6] = similarity[6, 5] = 2   # radio vs beber

sns.heatmap(similarity, ax=axes[1], cmap='YlOrRd',
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            annot=True, fmt='.0f', cbar_kws={'label': 'Similitud semántica'})
axes[1].set_title('Similitud Semántica entre Clases\n(mayor valor = mayor probabilidad de confusión)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Clase', fontsize=11)
axes[1].set_ylabel('Clase', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_balance_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_balance_analysis.png')

## 9. Análisis de Histogramas de Color Promedio por Clase

In [ ]:
# Calcular estadísticas de color (R, G, B) por clase
print('Calculando estadísticas de color por clase...')

color_stats = {}
for cls_id in range(5):
    cls_imgs = images[labels == cls_id].astype(np.float32)
    stats = {}
    for ch in range(3):
        ch_data = cls_imgs[:, :, :, ch].flatten()
        stats[ch] = {'mean': np.mean(ch_data), 'std': np.std(ch_data),
                     'median': np.median(ch_data)}
    color_stats[cls_id] = stats

# Tabla de resumen
print(f'{'Clase':<35} {"R_med":>6} {"G_med":>6} {"B_med":>6} {"R_std":>6} {"G_std":>6} {"B_std":>6}')
print('-' * 72)
for cls_id in range(5):
    cs = color_stats[cls_id]
    print(f'{CLASS_NAMES[cls_id]:<35} '
          f'{cs[0]["mean"]:>6.1f} {cs[1]["mean"]:>6.1f} {cs[2]["mean"]:>6.1f} '
          f'{cs[0]["std"]:>6.1f} {cs[1]["std"]:>6.1f} {cs[2]["std"]:>6.1f}')

In [ ]:
# Histogramas de color por clase (5 clases × 3 canales)
fig, axes = plt.subplots(5, 3, figsize=(14, 34))
fig.suptitle('Histogramas de Color por Clase y Canal (R, G, B)',
             fontsize=15, fontweight='bold', y=1.005)

ch_colors = ['red', 'green', 'blue']
ch_labels = ['Canal Rojo (R)', 'Canal Verde (G)', 'Canal Azul (B)']

for cls_id in range(5):
    cls_imgs = images[labels == cls_id]
    for ch in range(3):
        ax   = axes[cls_id, ch]
        data = cls_imgs[:, :, :, ch].flatten()
        ax.hist(data, bins=50, color=ch_colors[ch], alpha=0.7, density=True)
        ax.axvline(np.mean(data), color='black', linestyle='--', linewidth=1.5,
                  label=f'μ={np.mean(data):.0f}')
        ax.set_xlim(0, 255)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(alpha=0.3)
        ax.set_yticks([])
        if cls_id == 0:
            ax.set_title(ch_labels[ch], fontsize=11, fontweight='bold')
        if ch == 0:
            ax.set_ylabel(f'c{cls_id}', fontsize=5, fontweight='bold', rotation=0, labelpad=22)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_color_histograms.png'), bbox_inches='tight', dpi=50)
plt.show()
print('Guardado: fig_color_histograms.png')

In [ ]:
# Medias de intensidad RGB por clase (barras agrupadas)
fig, ax = plt.subplots(figsize=(14, 6))

x     = np.arange(5)
width = 0.25
means = [[color_stats[i][ch]['mean'] for i in range(5)] for ch in range(3)]

ax.bar(x - width, means[0], width, label='Canal R', color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x,         means[1], width, label='Canal G', color='#2ecc71', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x + width, means[2], width, label='Canal B', color='#3498db', alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Clase', fontsize=12)
ax.set_ylabel('Intensidad media (0–255)', fontsize=12)
ax.set_title('Intensidad Media de Canales R, G, B por Clase', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_SHORT)
ax.legend(fontsize=11)
ax.set_ylim(0, 295)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_color_means.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_color_means.png')

## 10. Análisis de Brillo y Contraste

In [ ]:
# Brillo (luminosidad percibida) y contraste (desviación estándar de grises) por imagen
brightness_by_class = {}
contrast_by_class   = {}

for cls_id in range(5):
    cls_imgs = images[labels == cls_id].astype(np.float32)
    # Luma según rec. ITU-R BT.601
    gray = 0.299*cls_imgs[:,:,:,0] + 0.587*cls_imgs[:,:,:,1] + 0.114*cls_imgs[:,:,:,2]
    brightness_by_class[cls_id] = np.mean(gray, axis=(1, 2))
    contrast_by_class[cls_id]   = np.std(gray,  axis=(1, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot de brillo
data_b = [brightness_by_class[i] for i in range(5)]
bp1 = axes[0].boxplot(data_b, labels=CLASS_SHORT, patch_artist=True)
for patch, color in zip(bp1['boxes'], CLASS_COLORS_VIS):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[0].set_xlabel('Clase', fontsize=12)
axes[0].set_ylabel('Brillo medio (0–255)', fontsize=12)
axes[0].set_title('Distribución de Brillo por Clase', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.4)

# Boxplot de contraste
data_c = [contrast_by_class[i] for i in range(5)]
bp2 = axes[1].boxplot(data_c, labels=CLASS_SHORT, patch_artist=True)
for patch, color in zip(bp2['boxes'], CLASS_COLORS_VIS):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[1].set_xlabel('Clase', fontsize=12)
axes[1].set_ylabel('Contraste (desv. estándar)', fontsize=12)
axes[1].set_title('Distribución de Contraste por Clase', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_brightness_contrast.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Guardado: fig_brightness_contrast.png')

## 11. Conclusiones del EDA

In [ ]:
# Resumen de hallazgos del EDA
print('='*65)
print('CONCLUSIONES DEL ANÁLISIS EXPLORATORIO DE DATOS (EDA)')
print('='*65)
print()
print(f'1. DATASET UTILIZADO: {data_source}')
print(f'   • {len(images)} imágenes totales / {len(set(labels.tolist()))} clases')
print(f'   • Tamaño uniforme: {images[0].shape[1]}x{images[0].shape[0]} px, 3 canales RGB')
print(f'   • Memoria: {images.nbytes/(1024**2):.1f} MB en RAM')
print()
print(f'2. BALANCE DE CLASES')
print(f'   • Ratio de desbalance: {imbalance_ratio:.2f}x (ideal = 1.00x)')
print(f'   • Coeficiente de variación: {cv:.1f}%')
decision = 'BIEN BALANCEADO' if cv < 10 else ('MODERADAMENTE BALANCEADO' if cv < 25 else 'DESBALANCEADO')
print(f'   • Veredicto: {decision}')
print()
print('3. CARACTERÍSTICAS VISUALES RELEVANTES')
print('   • Las clases presentan diferentes distribuciones de color medias')
print('   • c1/c3 (texto) y c2/c4 (teléfono) son pares semánticamente similares')
print('   • c0 (conducción segura) tiene el patrón visual más diferenciable')
print('   • c8 (cabello) y c7 (alcanzar) comparten movimientos corporales similares')
print()
print('4. RETOS DE CLASIFICACIÓN ANTICIPADOS')
print('   • Alta similitud entre clases 1-2 y 3-4 (uso del teléfono)')
print('   • Variabilidad inter-individuo: distintos conductores, iluminación, ángulos')
print('   • Necesidad de características de alto nivel (Transfer Learning recomendado)')
print()
print('5. RECOMENDACIONES PARA EL ENTRENAMIENTO')
print('   • Usar ResNet18 preentrenado en ImageNet (Transfer Learning)')
print('   • Aplicar data augmentation: flip, rotación, color jitter')
print('   • Monitorear matriz de confusión para clases similares')
print('   • Normalizar con medias/std de ImageNet para aprovechar pesos preentrenados')
print('='*65)

In [ ]:
# Listado final de artefactos generados
figuras = [
    'fig_class_distribution.png',
    'fig_examples_grid.png',
    'fig_size_stats.png',
    'fig_balance_analysis.png',
    'fig_color_histograms.png',
    'fig_color_means.png',
    'fig_brightness_contrast.png'
]

print('Artefactos generados por este notebook:')
for fname in figuras:
    fpath = os.path.join(OUTPUT_DIR, fname)
    ok    = os.path.exists(fpath)
    size  = os.path.getsize(fpath)/1024 if ok else 0
    mark  = 'OK' if ok else 'FALTANTE'
    print(f'  [{mark}] {fname} ({size:.1f} KB)')

print('\nEDA completado exitosamente.')